# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [ ]:
MODEL = "gemini-3.5-flash"

In [ ]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
You can look up ticket prices using get_ticket_price, 
and you can update or set ticket prices using set_ticket_price 
when the user asks you to change or add a price.
Always be accurate. If you don't know the answer, say so.
"""

### Using Tools

In [ ]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"

def set_ticket_price(destination_city, price):
    print(f"Tool called to set price for city {destination_city} to {price}")
    ticket_prices[destination_city.lower()] = price
    return f"The price of a ticket to {destination_city} has been set to {price}"


In [ ]:
get_ticket_price("London")

In [ ]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city whose ticket price should be set",
            },
            "price": {
                "type": "string",
                "description": "The new ticket price, e.g. '$499'",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [ ]:
# And this is included in a list of tools:

tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function}
]

In [ ]:
tools

### Getting OpenAI to use our Tool

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        tool_responses = handle_tool_call(assistant_message)
        messages.append(assistant_message)
        messages.extend(tool_responses)   # extend, not append, since it's a list now
        response = openai.chat.completions.create(model=MODEL, messages=messages)

    return response.choices[0].message.content

In [ ]:
def handle_tool_call(message):
    responses = []
    for tool_call in message.tool_calls:
        arguments = json.loads(tool_call.function.arguments)

        if tool_call.function.name == "get_ticket_price":
            city = arguments.get('destination_city')
            result = get_ticket_price(city)

        elif tool_call.function.name == "set_ticket_price":
            city = arguments.get('destination_city')
            price = arguments.get('price')
            result = set_ticket_price(city, price)

        else:
            result = "Unknown tool"

        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()